# 肝脏 CT 数据探索

本 notebook 用于探索 Medical Segmentation Decathlon Task03_Liver 数据集。

## 数据集说明

Medical Segmentation Decathlon（MSD）的 **Task03_Liver** 子任务提供腹部 CT 体数据与配套分割标签：

- `imagesTr/`：训练用 CT 图像（`.nii.gz`，3D 体数据）
- `labelsTr/`：训练用标签（0=背景，1=肝脏，2=肿瘤）
- `imagesTs/`：测试 CT 图像

**运行前必须先下载数据**：请将 Task03_Liver 解压到 `./data/Task03_Liver` 目录（即包含 `imagesTr/`、`labelsTr/` 子目录）。若数据未下载，后续 `glob` 结果将为空，可视化单元格会报错。

In [ ]:
# 导入数据探索常用的基础库
import os
import glob
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

%matplotlib inline

In [ ]:
# 设置数据根目录；按实际路径修改此处
# 期望结构：DATA_ROOT/imagesTr/*.nii.gz 与 DATA_ROOT/labelsTr/*.nii.gz
DATA_ROOT = './data/Task03_Liver'

In [ ]:
# 用 glob 列出训练图像文件并统计数量
image_files = sorted(glob.glob(os.path.join(DATA_ROOT, 'imagesTr', '*.nii.gz')))
print(f'找到 {len(image_files)} 个训练图像文件')

# TODO: 若数量为 0，说明数据尚未下载到 ./data/Task03_Liver，请先下载数据再运行本单元格。

In [ ]:
# 加载第一个样本的图像与标签，打印基本信息
# nib.load 返回 nibabel Image 对象，需用 get_fdata() 取体素数组
if len(image_files) == 0:
    raise FileNotFoundError('未找到图像文件，请先下载 Task03_Liver 数据到 ./data/Task03_Liver')

img_path = image_files[0]
label_path = img_path.replace('imagesTr', 'labelsTr')  # 路径推断对应标签

img = nib.load(img_path)
lbl = nib.load(label_path)

img_data = img.get_fdata()
lbl_data = lbl.get_fdata()

# shape: (D, H, W) 体素维度；spacing: (sx, sy, sz) 体素物理间距(mm)；dtype: 数据精度
print('图像 shape :', img_data.shape)
print('图像 spacing:', img.header.get_zooms()[:3])
print('图像 dtype :', img_data.dtype)
print('标签 shape :', lbl_data.shape)
print('标签 spacing:', lbl.header.get_zooms()[:3])
print('标签 dtype :', lbl_data.dtype)
print('标签取值集合:', np.unique(lbl_data))

In [ ]:
# 可视化某一轴向切片（axial slice）
# "轴向切片"指沿身体头-脚方向（通常对应数组第 2/3 维的某一固定层）截取的 2D 图像。
# 这里取中间一层，左边显示 CT 灰度，右边叠加标签（彩色）。

In [ ]:
mid = img_data.shape[2] // 2  # 取第 3 个维度（W）的中段切片作为 axial 切片示例
slice_img = img_data[:, :, mid]
slice_lbl = lbl_data[:, :, mid]

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(np.rot90(slice_img), cmap='gray')
axes[0].set_title(f'CT axial slice (z={mid})')
axes[0].axis('off')

# 在 CT 上叠加半透明标签：1=肝脏(红)、2=肿瘤(黄)
axes[1].imshow(np.rot90(slice_img), cmap='gray')
masked = np.ma.masked_where(slice_lbl == 0, slice_lbl)
axes[1].imshow(np.rot90(masked), cmap='autumn', alpha=0.6)
axes[1].set_title('CT + 标签叠加')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## TODO：继续探索

请在此探索更多内容，例如：
- 浏览不同切片（z 轴各层）观察肝脏与肿瘤出现的位置；
- 统计各类别体素占比（背景 / 肝脏 / 肿瘤），体会类别不平衡问题；
- 绘制全部样本的 spacing 分布，理解为什么需要重采样到统一体素间距；
- 观察 CT HU 值范围，思考为什么做 CT 窗归一化。